# Hot Rolling Defect Detection - Recall-First Binary Classification

Goal: detect all defective coils (`Y = 1`) while trying to keep precision above 90% and false positives below 10%.

Important validation note: with this dataset and the installed libraries, honest 5-fold cross-validation did **not** find a model/threshold that achieves all three constraints at once. The notebook therefore selects the best available recall-first fallback: the threshold with 100% validation recall and maximum precision. This is deliberately conservative because false negatives are the highest-cost error.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

RANDOM_STATE = 42
DATA_DIR = 'dataset'

In [ ]:
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print('train:', train.shape)
print('test:', test.shape)
print('sample_submission:', sample_submission.shape)
train.head()

In [ ]:
print(train.info())
print('
Target counts:')
print(train['Y'].value_counts())
print('
Target ratio:')
print(train['Y'].value_counts(normalize=True))
print('
Missing values in train:', train.isna().sum().sum())
print('Missing values in test:', test.isna().sum().sum())
print('
Duplicate CoilID in train:', train['CoilID'].duplicated().sum())
print('Duplicate CoilID in test:', test['CoilID'].duplicated().sum())

In [ ]:
ax = train['Y'].value_counts().sort_index().plot(kind='bar', title='Class Balance')
ax.set_xlabel('Y')
ax.set_ylabel('count')
plt.show()

train.drop(columns=['CoilID']).describe().T

In [ ]:
# Drop CoilID from model features but preserve test CoilID for submission.
X = train.drop(columns=['Y', 'CoilID'])
y = train['Y'].astype(int)
X_test = test.drop(columns=['CoilID'])
test_coil_id = test['CoilID']

print(X.shape, y.shape, X_test.shape)

## Models

`XGBoost`, `CatBoost`, `LightGBM`, and `imblearn` are optional. The code below uses them automatically if installed; otherwise it skips them. Sklearn models are always available here.

In [ ]:
models = {
    'Logistic Regression balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(max_iter=5000, class_weight='balanced', solver='liblinear', random_state=RANDOM_STATE))
    ]),
    'Random Forest balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(n_estimators=600, class_weight='balanced_subsample', max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'ExtraTrees balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', ExtraTreesClassifier(n_estimators=700, class_weight='balanced', max_features='sqrt', random_state=RANDOM_STATE, n_jobs=-1))
    ]),
    'GradientBoosting': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', GradientBoostingClassifier(n_estimators=250, learning_rate=0.04, max_depth=2, random_state=RANDOM_STATE))
    ]),
    'HistGradientBoosting balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', HistGradientBoostingClassifier(max_iter=300, learning_rate=0.04, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'GaussianNB': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', GaussianNB())
    ]),
    'QDA regularized': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', QuadraticDiscriminantAnalysis(reg_param=0.5))
    ]),
    'SVC RBF balanced': Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', RobustScaler()),
        ('model', SVC(C=1.0, gamma='scale', class_weight='balanced', probability=True, random_state=RANDOM_STATE))
    ]),
}

# Optional external gradient boosting models.
try:
    from xgboost import XGBClassifier
    models['XGBoost weighted'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', XGBClassifier(
            n_estimators=500, max_depth=3, learning_rate=0.03,
            subsample=0.9, colsample_bytree=0.9,
            scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
            eval_metric='logloss', random_state=RANDOM_STATE, n_jobs=-1
        ))
    ])
except Exception as e:
    print('XGBoost skipped:', e)

try:
    from catboost import CatBoostClassifier
    models['CatBoost balanced'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', CatBoostClassifier(iterations=700, depth=4, learning_rate=0.03, auto_class_weights='Balanced', verbose=False, random_seed=RANDOM_STATE))
    ])
except Exception as e:
    print('CatBoost skipped:', e)

try:
    from lightgbm import LGBMClassifier
    models['LightGBM balanced'] = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('model', LGBMClassifier(n_estimators=700, learning_rate=0.03, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1))
    ])
except Exception as e:
    print('LightGBM skipped:', e)

list(models)

In [ ]:
def evaluate_thresholds(y_true, proba, model_name):
    rows = []
    thresholds = np.r_[0, np.sort(np.unique(proba)), 1]
    auc = roc_auc_score(y_true, proba)
    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        rows.append({
            'model': model_name,
            'threshold': threshold,
            'accuracy': accuracy_score(y_true, pred),
            'precision': precision_score(y_true, pred, zero_division=0),
            'recall': recall_score(y_true, pred, zero_division=0),
            'f1': f1_score(y_true, pred, zero_division=0),
            'roc_auc': auc,
            'tn': tn, 'fp': fp, 'fn': fn, 'tp': tp,
            'fp_rate': fp / (fp + tn),
        })
    return pd.DataFrame(rows)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
all_thresholds = []
oof_predictions = {}

for name, model in models.items():
    print('Running:', name)
    try:
        proba = cross_val_predict(model, X, y, cv=cv, method='predict_proba')[:, 1]
    except Exception as e:
        print('Skipped:', name, e)
        continue
    oof_predictions[name] = proba
    all_thresholds.append(evaluate_thresholds(y, proba, name))

results = pd.concat(all_thresholds, ignore_index=True)

In [ ]:
# Competition condition: Recall = 100%, Precision > 90%, False Positive Rate < 10%.
valid = results[(results['recall'] == 1.0) & (results['precision'] > 0.90) & (results['fp_rate'] < 0.10)]

if len(valid):
    leaderboard = valid.sort_values(['precision', 'f1', 'roc_auc'], ascending=False)
    selection_reason = 'Meets all competition conditions.'
else:
    # Fallback: absolutely prioritize 100% recall, then maximize precision and minimize false positives.
    leaderboard = results[results['recall'] == 1.0].sort_values(['precision', 'fp_rate', 'f1', 'roc_auc'], ascending=[False, True, False, False])
    selection_reason = 'No CV model met all conditions; selected best 100% recall fallback.'

best_row = leaderboard.iloc[0]
print(selection_reason)
display(leaderboard.head(15))
print('
Selected model:', best_row['model'])
print('Selected threshold:', best_row['threshold'])

In [ ]:
# Also show what happens when precision is forced above 90%.
precision_first = results[results['precision'] >= 0.90].sort_values(['recall', 'precision', 'f1', 'roc_auc'], ascending=False)
display(precision_first.head(15))

In [ ]:
best_model_name = best_row['model']
best_threshold = float(best_row['threshold'])
best_oof_proba = oof_predictions[best_model_name]
best_oof_pred = (best_oof_proba >= best_threshold).astype(int)

print(classification_report(y, best_oof_pred, digits=4, zero_division=0))
cm = confusion_matrix(y, best_oof_pred, labels=[0, 1])
print('Confusion matrix [[TN, FP], [FN, TP]]:')
print(cm)

ConfusionMatrixDisplay(cm, display_labels=['No Defect', 'Defect']).plot(values_format='d')
plt.title(f'{best_model_name} at threshold {best_threshold:.6g}')
plt.show()

RocCurveDisplay.from_predictions(y, best_oof_proba)
plt.title(f'OOF ROC: {best_model_name}')
plt.show()

In [ ]:
# Train selected model on full training data and generate final submission.
final_model = models[best_model_name]
final_model.fit(X, y)

test_proba = final_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_threshold).astype(int)

submission = pd.DataFrame({'CoilID': test_coil_id, 'Y': test_pred})
submission.to_csv('expected_submission.csv', index=False)

print(submission.shape)
print(submission['Y'].value_counts())
submission.head()

In [ ]:
# Optional: inspect highest-risk coils.
risk_table = pd.DataFrame({'CoilID': test_coil_id, 'defect_probability': test_proba, 'Y': test_pred})
risk_table.sort_values('defect_probability', ascending=False).head(30)